# Convert filenames to basenames using `ConversionTable.xlsx`

This notebook reads a list of filenames, looks up corresponding basenames in `ConversionTable.xlsx`, and produces a comma-separated list of basenames with **commas and no spaces**.

Expected files in the same folder as this notebook:

- `filenames to_convert.txt` — input list of filenames, one per line or comma-separated
- `ConversionTable.xlsx` — Excel table with columns `basename` and `filename`


In [2]:
from pathlib import Path
from decimal import Decimal, InvalidOperation
import re
import pandas as pd

# --- Input / output paths ---
BASE_DIR = Path.cwd()

FILENAME_LIST_PATH = BASE_DIR / 'filenames to_convert.txt'
CONVERSION_TABLE_PATH = BASE_DIR / 'ConversionTable.xlsx'

OUTPUT_BASENAMES_TXT = BASE_DIR / 'converted_basenames_comma_list.txt'
OUTPUT_RESULT_CSV = BASE_DIR / 'converted_filenames_to_basenames.csv'

# Usually Zive basenames are written with 3 digits after the decimal point, e.g. 1636480.010
BASENAME_DECIMALS = 3

print('Working folder:', BASE_DIR)
print('Filename list:', FILENAME_LIST_PATH)
print('Conversion table:', CONVERSION_TABLE_PATH)


Working folder: /home/kesju/DI/2025_ZIVEO/PROJECT_TRAIN_UNET/1_PREPARE_TRAIN_UNET_DATA/CONVERT_ZIVE_TO_NPY
Filename list: /home/kesju/DI/2025_ZIVEO/PROJECT_TRAIN_UNET/1_PREPARE_TRAIN_UNET_DATA/CONVERT_ZIVE_TO_NPY/filenames to_convert.txt
Conversion table: /home/kesju/DI/2025_ZIVEO/PROJECT_TRAIN_UNET/1_PREPARE_TRAIN_UNET_DATA/CONVERT_ZIVE_TO_NPY/ConversionTable.xlsx


In [3]:
def normalize_filename(value) -> str:
    """Normalize filename values for lookup: strip spaces and keep only the file name."""
    if pd.isna(value):
        return ''
    s = str(value).strip().strip('\"').strip("'")
    if not s:
        return ''
    return Path(s).name


def filename_lookup_keys(value) -> list[str]:
    """Return several possible keys so lookup works with 1001_6 and 1001_6.npy."""
    fn = normalize_filename(value)
    if not fn:
        return []
    stem = Path(fn).stem
    keys = [fn, stem]
    if not fn.lower().endswith('.npy'):
        keys.append(fn + '.npy')
    # preserve order, remove duplicates
    return list(dict.fromkeys(keys))


def format_basename(value, decimals: int = 3) -> str:
    """Format numeric basenames with fixed decimal places; keep non-numeric basenames unchanged."""
    if pd.isna(value):
        return ''
    s = str(value).strip().strip('\"').strip("'")
    if not s:
        return ''
    # Remove accidental .npy/.json suffixes if they appear in basename column
    s = re.sub(r'\.(npy|json)$', '', s, flags=re.IGNORECASE)
    try:
        d = Decimal(s)
        return f'{d:.{decimals}f}'
    except (InvalidOperation, ValueError):
        return s


In [4]:
# --- Read input filename list ---
if not FILENAME_LIST_PATH.exists():
    raise FileNotFoundError(f'Input filename list not found: {FILENAME_LIST_PATH}')

raw_text = FILENAME_LIST_PATH.read_text(encoding='utf-8-sig')

# Accept one-per-line, comma-separated, semicolon-separated, or whitespace-separated lists
input_filenames = [x.strip() for x in re.split(r'[,;\s]+', raw_text) if x.strip()]

print(f'Read {len(input_filenames)} filenames')
input_filenames[:10]


Read 35 filenames


['1001_6.npy',
 '1001_8.npy',
 '1004_0.npy',
 '1005_0.npy',
 '1005_4.npy',
 '1006_1.npy',
 '1007_1.npy',
 '1008_0.npy',
 '1008_12.npy',
 '1009_12.npy']

In [5]:
# --- Read conversion table ---
if not CONVERSION_TABLE_PATH.exists():
    raise FileNotFoundError(f'Conversion table not found: {CONVERSION_TABLE_PATH}')

conv = pd.read_excel(CONVERSION_TABLE_PATH, dtype=str)
conv.columns = [str(c).strip() for c in conv.columns]

required_cols = {'basename', 'filename'}
missing_cols = required_cols - set(conv.columns)
if missing_cols:
    raise ValueError(f'ConversionTable.xlsx must contain columns {required_cols}. Missing: {missing_cols}. Found: {list(conv.columns)}')

conv = conv[['basename', 'filename']].copy()
conv['basename_norm'] = conv['basename'].map(format_basename)

# Build a robust lookup dictionary: keys include filename with and without .npy suffix
lookup = {}
duplicates = []

for _, row in conv.iterrows():
    basename = row['basename_norm']
    for key in filename_lookup_keys(row['filename']):
        if key in lookup and lookup[key] != basename:
            duplicates.append((key, lookup[key], basename))
        lookup[key] = basename

print(f'Read {len(conv)} rows from ConversionTable.xlsx')
print(f'Created {len(lookup)} filename lookup keys')

if duplicates:
    print('WARNING: duplicate filename keys with different basenames found:')
    for item in duplicates[:20]:
        print(item)
    if len(duplicates) > 20:
        print(f'... and {len(duplicates) - 20} more')

conv.head()


Read 1094 rows from ConversionTable.xlsx
Created 2188 filename lookup keys


,basename,filename,basename_norm
0,1626941.468,1000_1,1626941.468
1,1632923.661,1000_2,1632923.661
2,1621694.321,1001_0,1621694.321
3,1626330.744,1001_1,1626330.744
4,1626924.927,1001_2,1626924.927


In [6]:
# --- Convert filenames to basenames ---
rows = []
missing = []

for original in input_filenames:
    found_basename = None
    used_key = None
    for key in filename_lookup_keys(original):
        if key in lookup:
            found_basename = lookup[key]
            used_key = key
            break

    if found_basename is None:
        missing.append(original)

    rows.append({
        'filename': original,
        'basename': found_basename if found_basename is not None else '',
        'lookup_key_used': used_key if used_key is not None else '',
        'status': 'OK' if found_basename is not None else 'MISSING'
    })

result = pd.DataFrame(rows)

if missing:
    print(f'WARNING: {len(missing)} filenames were not found in ConversionTable.xlsx:')
    for x in missing:
        print('  -', x)
else:
    print('All filenames were converted successfully.')

result


All filenames were converted successfully.


,filename,basename,lookup_key_used,status
0,1001_6.npy,1670188.752,1001_6.npy,OK
1,1001_8.npy,1670183.201,1001_8.npy,OK
2,1004_0.npy,1630715.197,1004_0.npy,OK
3,1005_0.npy,1630733.908,1005_0.npy,OK
4,1005_4.npy,1630736.382,1005_4.npy,OK
5,1006_1.npy,1630797.676,1006_1.npy,OK
6,1007_1.npy,1630959.214,1007_1.npy,OK
7,1008_0.npy,1630771.589,1008_0.npy,OK
8,1008_12.npy,1630758.549,1008_12.npy,OK
9,1009_12.npy,1631057.107,1009_12.npy,OK


In [7]:
# --- Make comma-separated basename list with no spaces ---
basenames = result.loc[result['status'].eq('OK'), 'basename'].tolist()
basename_comma_list = ','.join(basenames)

print(basename_comma_list)

OUTPUT_BASENAMES_TXT.write_text(basename_comma_list, encoding='utf-8')
result.to_csv(OUTPUT_RESULT_CSV, index=False, encoding='utf-8-sig')

print('\nSaved comma-separated basename list to:', OUTPUT_BASENAMES_TXT)
print('Saved conversion result table to:', OUTPUT_RESULT_CSV)


1670188.752,1670183.201,1630715.197,1630733.908,1630736.382,1630797.676,1630959.214,1630771.589,1630758.549,1631057.107,1631056.479,1630869.824,1630890.629,1630878.738,1631074.415,1631080.143,1630995.587,1630941.751,1631280.395,1631087.069,1631085.830,1631036.053,1631013.491,1632340.564,1632252.683,1632471.213,1632731.258,1632728.750,1633044.261,1633041.936,1633047.767,1646777.284,1646779.781,1743959.255,1743958.630

Saved comma-separated basename list to: /home/kesju/DI/2025_ZIVEO/PROJECT_TRAIN_UNET/1_PREPARE_TRAIN_UNET_DATA/CONVERT_ZIVE_TO_NPY/converted_basenames_comma_list.txt
Saved conversion result table to: /home/kesju/DI/2025_ZIVEO/PROJECT_TRAIN_UNET/1_PREPARE_TRAIN_UNET_DATA/CONVERT_ZIVE_TO_NPY/converted_filenames_to_basenames.csv


## Notes

- The final comma-separated list is saved to `converted_basenames_comma_list.txt`.
- A detailed table with `filename`, `basename`, and `status` is saved to `converted_filenames_to_basenames.csv`.
- If a filename is missing from `ConversionTable.xlsx`, it is shown with status `MISSING`.
